# Practical Session 04c &mdash; Evaluating a classifier

**Companion to Lecture&nbsp;04.** The third of four notebooks. A trained classifier outputs
**probabilities**; turning them into decisions and judging them well is its own skill. We build the
**confusion matrix**, **precision/recall/F1**, the **ROC curve** and **AUC** from scratch (checked
against scikit-learn), see the **threshold** as a tunable knob, and watch **accuracy lie** under
class imbalance &mdash; on a real medical dataset.

| # | Notebook | Topic |
|---|----------|-------|
| 04a | the model | the sigmoid, odds/log-odds, the decision boundary |
| 04b | training from scratch | cross-entropy, the gradient, gradient descent, convexity, $L_2$ |
| **04c** | **evaluation** | **confusion matrix, precision/recall/F1, ROC/AUC, thresholds, imbalance (this notebook)** |
| 04d | multiclass | softmax from scratch, cross-entropy, decision regions, digits |

> ⭐ **Key idea.** A single accuracy number hides *how* a classifier is right or wrong. The
> **confusion matrix** splits errors into false positives and false negatives; **precision** and
> **recall** trade off as you move the **threshold**; and **ROC/AUC** measure ranking quality
> independent of any one threshold.

*Run each cell (Shift+Enter). Self-contained and offline (scikit-learn ships the dataset).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nb_utils import use_style
use_style("notes")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Real data. We define the POSITIVE class as "malignant" (the thing we want to catch),
# so recall = fraction of cancers detected.
data = load_breast_cancer()
X, y = data.data, (data.target == 0).astype(int)   # 1 = malignant, 0 = benign
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]               # P(malignant) on the test set
print("test set:", len(yte), "cases,", int(yte.sum()), "malignant")
print("ready")

## 1. Probabilities &rarr; decisions: the confusion matrix

Threshold the probability at $0.5$ to get a hard label, then cross-tabulate predictions against
truth. Every test case falls into one of four boxes:

* **TP** true positive (caught a malignancy), **TN** true negative,
* **FP** false positive (false alarm), **FN** false negative (**missed a cancer** &mdash; the costly one).

In [ ]:
def confusion(y_true, y_pred):
    TP = int(((y_pred == 1) & (y_true == 1)).sum())
    TN = int(((y_pred == 0) & (y_true == 0)).sum())
    FP = int(((y_pred == 1) & (y_true == 0)).sum())
    FN = int(((y_pred == 0) & (y_true == 1)).sum())
    return TP, FP, FN, TN

pred = (proba >= 0.5).astype(int)
TP, FP, FN, TN = confusion(yte, pred)
print(f"TP={TP}  FP={FP}\nFN={FN}  TN={TN}")

from sklearn.metrics import confusion_matrix
print("\nmatches sklearn confusion_matrix:",
      np.array_equal(np.array([[TN, FP], [FN, TP]]), confusion_matrix(yte, pred)))

fig, ax = plt.subplots(figsize=(3.6, 3.2))
M = np.array([[TN, FP], [FN, TP]])
ax.imshow(M, cmap="Blues")
for (i, j), v in np.ndenumerate(M):
    ax.text(j, i, v, ha="center", va="center", fontsize=13,
            color="white" if v > M.max()/2 else "black")
ax.set_xticks([0, 1], ["pred 0", "pred 1"]); ax.set_yticks([0, 1], ["true 0", "true 1"])
ax.set_title("confusion matrix"); plt.show()

## 2. Accuracy, precision, recall, F1 &mdash; from scratch

From the four counts:

$$\text{accuracy}=\frac{TP+TN}{\text{all}},\quad
\text{precision}=\frac{TP}{TP+FP},\quad
\text{recall}=\frac{TP}{TP+FN},\quad
F_1=\frac{2\,PR}{P+R}.$$

**Precision** = "when I raise the alarm, how often am I right?" **Recall** = "of all real
positives, how many did I catch?" $F_1$ is their harmonic mean.

In [ ]:
def metrics(y_true, y_pred):
    TP, FP, FN, TN = confusion(y_true, y_pred)
    acc = (TP + TN) / len(y_true)
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    rec = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return acc, prec, rec, f1

acc, prec, rec, f1 = metrics(yte, pred)
print(f"accuracy={acc:.3f}  precision={prec:.3f}  recall={rec:.3f}  F1={f1:.3f}")

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("matches sklearn:",
      np.allclose([acc, prec, rec, f1],
                  [accuracy_score(yte, pred), precision_score(yte, pred),
                   recall_score(yte, pred), f1_score(yte, pred)]))

## 3. The threshold is a knob: precision vs recall

$0.5$ is just a default. Lower the threshold and you catch more positives (**recall up**) but raise
more false alarms (**precision down**); raise it and the trade goes the other way. In cancer
screening a missed case (FN) is far worse than a false alarm (FP), so we might deliberately lower
the threshold.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
precs, recs = [], []
for t in thresholds:
    _, p, r, _ = metrics(yte, (proba >= t).astype(int))
    precs.append(p); recs.append(r)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.2, 3.4))
a1.plot(thresholds, precs, label="precision", color="#0072B2")
a1.plot(thresholds, recs, label="recall", color="#D55E00")
a1.axvline(0.5, color="0.6", ls=":"); a1.set_xlabel("threshold"); a1.set_ylabel("score")
a1.set_title("precision & recall vs threshold"); a1.legend(fontsize=8)
a2.plot(recs, precs, color="#009E73"); a2.set_xlabel("recall"); a2.set_ylabel("precision")
a2.set_title("precision-recall trade-off"); a2.set_xlim(0, 1.02); a2.set_ylim(0, 1.02)
plt.show()

# highest threshold that still catches >= 98% of cancers (fewest false alarms at that recall)
ok = [t for t, r in zip(thresholds, recs) if r >= 0.98]
print("highest threshold with recall >= 0.98:", round(max(ok), 2) if ok else "n/a")

## 4. ROC curve and AUC (from scratch)

Sweeping the threshold traces the **ROC curve**: true-positive rate (recall) against
false-positive rate. The **area under it (AUC)** summarizes ranking quality in one number,
*independent of any threshold*: AUC $=$ the probability the model scores a random positive above a
random negative ($0.5$ = coin flip, $1.0$ = perfect). We build both from scratch.

In [ ]:
def roc_curve_scratch(y_true, scores):
    order = np.argsort(-scores)          # high score first
    y = y_true[order]
    P, N = y.sum(), len(y) - y.sum()
    tpr = np.concatenate([[0], np.cumsum(y) / P])
    fpr = np.concatenate([[0], np.cumsum(1 - y) / N])
    return fpr, tpr

def auc_scratch(y_true, scores):
    # Mann-Whitney form: mean rank of positives, ties handled by average ranks
    from scipy.stats import rankdata
    r = rankdata(scores)
    n_pos, n_neg = y_true.sum(), (1 - y_true).sum()
    return (r[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

fpr, tpr = roc_curve_scratch(yte, proba)
auc = auc_scratch(yte, proba)

from sklearn.metrics import roc_auc_score
print(f"our AUC = {auc:.4f}   sklearn AUC = {roc_auc_score(yte, proba):.4f}")
print("match:", np.isclose(auc, roc_auc_score(yte, proba)))

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="0.6", label="chance")
ax.set_xlabel("false-positive rate"); ax.set_ylabel("true-positive rate (recall)")
ax.set_title("ROC curve"); ax.legend(fontsize=8, loc="lower right"); plt.show()

## 5. Why accuracy lies: class imbalance

When one class is rare, a lazy model that **always predicts the majority** scores high accuracy
while catching **none** of the positives. Precision/recall/F1 (and AUC) expose this; accuracy does
not. We build a 5%-positive dataset and compare a trivial baseline to logistic regression.

In [ ]:
from sklearn.datasets import make_classification
Xi, yi = make_classification(n_samples=4000, n_features=8, weights=[0.95, 0.05],
                             n_informative=4, random_state=0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xi, yi, test_size=0.4, random_state=0, stratify=yi)
print(f"positives in test: {yte2.mean():.1%}")

# (a) trivial 'always predict 0' baseline
base = np.zeros_like(yte2)
# (b) logistic regression with class_weight balanced (tell it the rare class matters)
lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=5000, class_weight="balanced")).fit(Xtr2, ytr2)
pred_lr = lr.predict(Xte2)

for name, pr in [("always-0 baseline", base), ("logistic (balanced)", pred_lr)]:
    a, p, r, f = metrics(yte2, pr)
    print(f"{name:22s} acc={a:.3f}  precision={p:.3f}  recall={r:.3f}  F1={f:.3f}")
print(f"\nlogistic AUC = {roc_auc_score(yte2, lr.predict_proba(Xte2)[:,1]):.3f}")

> ⚠️ **Watch out.** The always-0 baseline scores ~95% **accuracy** yet has **recall 0** &mdash; it
> catches no positives and its F1 is 0. Never report accuracy alone on imbalanced data; lead with
> recall/precision/F1 (or AUC), and consider `class_weight="balanced"` or a tuned threshold.

## Recap & exercises

**Recap.**
* The **confusion matrix** (TP/FP/FN/TN) is the source of every classification metric.
* **Precision** (alarms that were right) and **recall** (positives that were caught) trade off as the **threshold** moves; $F_1$ balances them. $0.5$ is only a default.
* **ROC/AUC** measure threshold-independent ranking quality; AUC $=$ P(score of a random positive $>$ a random negative). All matched scikit-learn.
* Under **class imbalance**, accuracy is misleading &mdash; a majority-only model looks great yet catches nothing. Use recall/precision/F1/AUC and `class_weight`/thresholds.

**Exercises.**
1. On the cancer data, pick the threshold from Section&nbsp;3 that reaches recall $\ge0.98$ and report the new confusion matrix. How many false alarms did catching those extra cancers cost?
2. Plot the **precision-recall curve** for the imbalanced dataset and compare its shape to the ROC curve. Which is more informative when positives are rare?
3. Remove `class_weight="balanced"` in Section&nbsp;5. What happens to recall on the rare class?
4. Compute **balanced accuracy** = (recall on class 1 + recall on class 0)/2 from scratch and compare to `sklearn.metrics.balanced_accuracy_score`.
5. Add a second model (e.g. no scaling) and compare AUCs. Does scaling matter here?

*Next:* **04d &mdash; beyond two classes**: the **softmax** generalization, its cross-entropy and
gradient, and multiclass decision regions on the digits dataset.